# 05 — MGSIM with the WestPilbara schedule: a fresh variogram at every level

Notebook 03 ran the `multigrid_sgsim` package the way it ships: **one global
variogram**, fitted once on the residuals vs the published bed, used at every
multigrid pass. The WestPilbara production run (see the notebook-04 addendum)
does something different at each level: the residual left after the previous
level gets its **own** normal-score transform and its **own** Gaussian variogram
— range shrinking with the level spacing — before SGSIM.

This notebook applies that schedule *inside the package*. Per level:

1. residual = data − current field at the conditioning nodes;
2. refit the NST and a Gaussian variogram on that residual (1% nugget stabilizer);
3. the package's own multigrid sampler picks the level's conditioning subset;
4. one `mgsim` pass at that level; the field absorbs the simulated residual.

Everything else is notebook 03: same conditioning set (datum shift, 0.25/99.75%
trim, checkerboard thinning), k = 12, radius 50 px, 98% NST clip, 2 km cosine
wall taper to the published bed, 20 m ice guard. Two deliberate deviations from
WestPilbara: a **fresh seed per member and per level** (WestPilbara reuses one
seed for all members), and a four-level ladder **12 / 6 / 3 / 1 px**
(1.8 km → 150 m) so the coarse level sits near the fitted ~2 km range —
WestPilbara's 4 / 2 / 1-cell ladder rescaled to our line spacing.

Needs the same packages as notebook 03; ~7 min per member, cached per seed in `real_cache_mgpl/`.

In [ ]:
import io, json, time, contextlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import skgstat as skg
from sklearn.preprocessing import QuantileTransformer
from multigrid_sgsim import mgsim, cluster_variogram
from multigrid_sgsim.sampling import subsample_dataframe

N_GRID, PX_M = 166, 150.0
SRC, DERIVED, FIGS = Path('../data/source_copies'), Path('../data/derived'), Path('../figures')

def read_dat(p):
    return np.loadtxt(p)[:, 2].reshape(N_GRID, N_GRID)

trend = read_dat(SRC / '2019cw428_bed_d25km_h1004_a0.792.dat')
surface = read_dat(SRC / '2019cw428_surface_d25km_h1004_a0.792.dat')

# conditioning data — identical preparation to notebooks 03/04
gz = np.load(DERIVED / 'gridded_bed_cw1951_150m.npz')
I0, I1 = json.loads(str(gz['meta']))['dem_sub_lattice']
radar = gz['bed_hm'][I0:I1 + 1, I0:I1 + 1]
datum_shift = float(np.nanmedian(radar - trend))
radar_aligned = radar - datum_shift
resid = radar_aligned - trend
lo, hi = np.nanpercentile(resid, [0.25, 99.75])
obs_mask = np.isfinite(resid) & (resid >= lo) & (resid <= hi)
chk = (np.add.outer(np.arange(N_GRID), np.arange(N_GRID)) % 2) == 0
cond_mask = obs_mask & chk                      # checkerboard-thinned, as nb 03
cy, cx = np.where(cond_mask)
cond_xy = np.column_stack([cx, cy]).astype(float)
cond_val = radar_aligned[cond_mask]
iy_all, ix_all = np.where(obs_mask)             # every trimmed obs node (metrics)
vals_all = radar_aligned[obs_mask]
gy, gx = np.mgrid[0:N_GRID, 0:N_GRID]

R3 = np.load(DERIVED / 'bed_realizations_cw1951.npz')
sgs_beds, taper_w = R3['beds'], R3['taper']
grf_beds = np.load(DERIVED / 'bed_realizations_cw1951_condgrf.npz')['beds']

print(f'datum shift {datum_shift:+.1f} m · {int(obs_mask.sum())} obs nodes · '
      f'{int(cond_mask.sum())} conditioning nodes after thinning · '
      f'residual std {resid[obs_mask].std():.1f} m')

In [ ]:
MG_LEVELS = [12, 6, 3, 1]          # px; 1.8 km -> 150 m
MAXLAG, N_LAGS = 60.0, 40          # variogram fit window, as nb 03
K_NEIGHBORS, RADIUS, CLIP_PCT = 12, 50.0, 98.0

def fit_level_vario(resid_obs, rng):
    """NST + Gaussian variogram fitted on the *current* residuals (<= 4000-pt
    subsample for the O(n^2) pairwise step). 1% nugget, range floored at 1.5 px."""
    nst = QuantileTransformer(output_distribution='normal',
                              n_quantiles=min(1000, len(resid_obs))).fit(resid_obs.reshape(-1, 1))
    df = pd.DataFrame({'x': cond_xy[:, 0], 'y': cond_xy[:, 1], 'cluster': 0,
                       'Nresidual': nst.transform(resid_obs.reshape(-1, 1)).ravel()})
    vsub = df.sample(n=min(4000, len(df)), random_state=int(rng.integers(2**31 - 1)))
    V = cluster_variogram(vsub, 'Nresidual', 0, MAXLAG, N_LAGS, 'gaussian')
    r, s, n = (float(v) for v in V.parameters)
    r = float(np.clip(r, 1.5, MAXLAG))
    return nst, [0.0, 0.01 * s, r, r, s, 'Gaussian'], V

def mgsim_perlevel(seed, trace=False):
    """One ensemble member: the package, one level per call, variogram refit
    between levels. Returns the field, a per-level info table, and (optionally)
    the per-level fields + variograms for the trace figures."""
    rng = np.random.default_rng(seed)
    field = trend.copy()
    info, stages = [], []
    for lev in MG_LEVELS:
        resid_obs = cond_val - field[cond_mask]
        nst, vario, V = fit_level_vario(resid_obs, rng)
        df_obs = pd.DataFrame({'x': cond_xy[:, 0], 'y': cond_xy[:, 1], 'residual': resid_obs})
        if lev > 1:   # the package's lattice sampler (variance-retry), matched back by (x, y)
            sub = subsample_dataframe(df_obs, column_for_sampling='residual',
                                      spacing=float(lev), random_state=rng)
            sel = np.zeros((N_GRID, N_GRID), bool)
            sel[sub['y'].astype(int).values, sub['x'].astype(int).values] = True
        else:
            sel = cond_mask.copy()
        df_grid = pd.DataFrame({'x': gx.ravel().astype(float), 'y': gy.ravel().astype(float),
                                'trend': field.ravel(), 'cluster': 0,
                                'value': np.where(sel, radar_aligned, np.nan).ravel()})
        df_grid['set'] = np.isfinite(df_grid['value']).astype(int)
        t0 = time.time()
        with contextlib.redirect_stdout(io.StringIO()):
            out = mgsim(mg_resols=[lev], df_xyvtcs=df_grid, vario=vario, nst_trans=nst,
                        seed=int(rng.integers(2**31 - 1)), xx='x', yy='y', kk='cluster',
                        num_points=K_NEIGHBORS, radius=RADIUS, sgs_or_krige='sgs',
                        clip_nst=True, clip_percentile=CLIP_PCT)
        field = out['newtrend'].values.reshape(N_GRID, N_GRID)
        info.append(dict(level_px=lev, n_cond=int(sel.sum()), resid_std_m=round(float(resid_obs.std()), 1),
                         range_km=round(vario[2] * PX_M / 1e3, 3), sill_nst=round(vario[4], 3),
                         seconds=round(time.time() - t0)))
        if trace:
            stages.append(dict(level=lev, field=field.copy(), bins=[float(b) for b in V.bins],
                               exp=[float(e) for e in V.experimental],
                               params=[vario[2], vario[4], vario[1]], resid_std=float(resid_obs.std())))
    return field, info, stages

In [ ]:
N_REAL = 10
CACHE = DERIVED / 'real_cache_mgpl'; CACHE.mkdir(exist_ok=True)
reals = np.full((N_REAL, N_GRID, N_GRID), np.nan, dtype=np.float32)
infos, stage_fields, stage_vario = {}, None, None
for i in range(N_REAL):
    f = CACHE / f'real_seed{i:02d}.npz'
    if f.exists():
        z = np.load(f)
        reals[i] = z['bed']; infos[i] = json.loads(str(z['info']))
        if 'stage_fields' in z.files:
            stage_fields, stage_vario = z['stage_fields'], json.loads(str(z['stage_vario']))
        print(f'seed {i}: cached'); continue
    t0 = time.time()
    field, info, stages = mgsim_perlevel(i, trace=(i == 0))
    reals[i] = field.astype(np.float32); infos[i] = info
    extra = {}
    if stages:
        stage_fields = np.array([s['field'] for s in stages], dtype=np.float32)
        stage_vario = [{k: v for k, v in s.items() if k != 'field'} for s in stages]
        extra = dict(stage_fields=stage_fields, stage_vario=json.dumps(stage_vario))
    np.savez_compressed(f, bed=reals[i], info=json.dumps(info), **extra)
    print(f'seed {i}: {time.time() - t0:.0f} s · ' +
          ' · '.join(f"L{d['level_px']}: r {d['range_km']:.2f} km, resid {d['resid_std_m']:.0f} m" for d in info))

In [ ]:
# --- the fresh variogram at each level ---------------------------------------
print('seed 0, level by level:')
print(pd.DataFrame(infos[0]).to_string(index=False))
rng_tab = pd.DataFrame({f'L{l} px': [infos[i][j]['range_km'] for i in range(N_REAL)]
                        for j, l in enumerate(MG_LEVELS)})
print('\nfitted range (km) across the 10 members:')
print(rng_tab.describe().loc[['mean', 'std', 'min', 'max']].round(2).to_string())

h = np.linspace(0, MAXLAG, 200)
fig, axs = plt.subplots(1, 4, figsize=(20, 4.4), constrained_layout=True)
for ax, sv in zip(axs, stage_vario):
    r, s, n = sv['params']
    ax.scatter(sv['bins'], sv['exp'], c='k', s=18, zorder=3, label='empirical (omni)')
    ax.plot(h, [skg.models.gaussian(hh, r, s, n) for hh in h], color='#444', lw=2, label='Gaussian fit, 1% nugget')
    ax.axvline(sv['level'], color='#c0392b', ls=':', lw=1.2, label=f"level spacing {sv['level']} px")
    ax.set_title(f"level {sv['level']} px · range {r * PX_M / 1e3:.2f} km · sill {s:.2f} · resid std {sv['resid_std']:.0f} m", fontsize=9)
    ax.set_xlabel('lag [px, 150 m]'); ax.grid(alpha=.3)
axs[0].set_ylabel('γ(h) [NST units]'); axs[0].legend(fontsize=8)
fig.suptitle('a fresh variogram at every level (seed 0): the residual left after each pass', fontsize=11)
fig.savefig(FIGS / 'mgpl_variograms.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# --- the cascade, seed 0 ------------------------------------------------------
vmin, vmax = np.percentile(trend, [2, 98])
fig, axs = plt.subplots(1, 5, figsize=(21, 4.4), constrained_layout=True)
panels = [('published bed (trend)', trend)] + [(f"+ level {sv['level']} px", fld) for sv, fld in zip(stage_vario, stage_fields)]
for ax, (t, b) in zip(axs, panels):
    im = ax.imshow(b, origin='lower', cmap='terrain', vmin=vmin, vmax=vmax)
    ax.set_title(t, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axs, shrink=0.8, label='bed z (m)')
fig.suptitle('MGSIM per-level cascade, seed 0: 1.8 km / 900 m / 450 m / 150 m passes, each with its own variogram', fontsize=11)
fig.savefig(FIGS / 'mgpl_cascade.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# --- taper + guard, then metrics and gallery vs nb 03 / nb 04 -----------------
MIN_ICE = 20.0
def apply_taper(b):
    return trend + taper_w * (b - trend)
mgpl_beds = np.array([apply_taper(b) for b in reals])
n_clip = int((mgpl_beds > surface - MIN_ICE).sum())
mgpl_beds = np.minimum(mgpl_beds, surface - MIN_ICE)
print(f'taper (nb-03 weights) + {MIN_ICE:.0f} m ice guard: {n_clip} node-clips')

core = slice(20, -20)
in_core = (ix_all >= 20) & (ix_all < N_GRID - 20) & (iy_all >= 20) & (iy_all < N_GRID - 20)
def rough(b):
    d2 = np.diff(b, n=2, axis=1)[core, core]
    return float(np.sqrt(np.mean(d2 ** 2)))
def metrics(beds, name):
    err = np.abs(beds[:, iy_all[in_core], ix_all[in_core]] - vals_all[in_core])
    return dict(method=name,
                rough_m=np.mean([rough(b) for b in beds]),
                dep_std_m=np.mean([np.std((b - trend)[core, core]) for b in beds]),
                cond_err_med_m=float(np.median(err)), cond_err_p95_m=float(np.percentile(err, 95)),
                ens_std_core_m=float(beds.std(0)[core, core].mean()))
ENS = [(sgs_beds, 'MGSIM package — global variogram (nb 03)', 'tab:orange'),
       (mgpl_beds, 'MGSIM package — per-level variogram (nb 05)', 'tab:green'),
       (grf_beds, 'conditional GRF (nb 04)', 'tab:blue')]
rows = [metrics(b, n) for b, n, _ in ENS]
print(f'\nreference: published-bed roughness {rough(trend):.1f} m · radar-resid std {resid[obs_mask].std():.1f} m\n')
print(pd.DataFrame(rows).round(1).to_string(index=False))

fig, axs = plt.subplots(2, 5, figsize=(21, 8.6), constrained_layout=True)
for row, (beds, label, _) in enumerate(ENS[:2]):
    for j in range(4):
        im = axs[row, j].imshow(beds[j], origin='lower', cmap='terrain', vmin=vmin, vmax=vmax)
        axs[row, j].set_title(f'{label} · seed {j}', fontsize=8)
    ims = axs[row, 4].imshow(beds.std(0), origin='lower', cmap='magma', vmin=0, vmax=80)
    axs[row, 4].set_title(f'{label} · ensemble std', fontsize=8)
    for ax in axs[row]:
        ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axs[:, :4], shrink=0.7, label='bed z (m)')
fig.colorbar(ims, ax=axs[:, 4], shrink=0.7, label='std (m)')
fig.savefig(FIGS / 'mgpl_gallery.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# --- spread vs distance-to-data, and the flowline cross-section --------------
P = np.column_stack([ix_all, iy_all]).astype(float)
tgt = np.column_stack([gx.ravel(), gy.ravel()]).astype(float)
dmap = cKDTree(P).query(tgt)[0].reshape(N_GRID, N_GRID)
bins = np.arange(0, 42, 2)
sig_data = resid[obs_mask].std()
fig, ax = plt.subplots(figsize=(8, 5))
for beds, name, c in ENS:
    sd = beds.std(0); bc, prof = [], []
    for b0, b1 in zip(bins[:-1], bins[1:]):
        sel = (dmap[core, core] >= b0) & (dmap[core, core] < b1)
        if sel.sum() > 30:
            bc.append((b0 + b1) / 2 * PX_M / 1000); prof.append(sd[core, core][sel].mean())
    ax.plot(bc, prof, '-o', ms=3, color=c, label=name)
ax.axhline(sig_data, ls=':', c='gray', label=f'radar-resid σ ({sig_data:.0f} m)')
ax.set_xlabel('distance to nearest radar cell (km)'); ax.set_ylabel('ensemble std (m)')
ax.set_title('where each ensemble puts its uncertainty'); ax.legend(fontsize=9)
fig.savefig(FIGS / 'mgpl_spread_vs_distance.png', dpi=110, bbox_inches='tight')
plt.show()

IY = 82
x_km = np.arange(N_GRID) * PX_M / 1000
fig, axs = plt.subplots(3, 1, figsize=(12, 10), sharex=True, sharey=True, constrained_layout=True)
for ax, (beds, name, c) in zip(axs, ENS):
    ax.fill_between(x_km, beds[:, IY, :].min(0), beds[:, IY, :].max(0), alpha=.3, color=c)
    ax.plot(x_km, beds[:, IY, :].mean(0), color=c, lw=1.2, label=f'{name}: mean, min–max of {N_REAL}')
    ax.plot(x_km, trend[IY], 'k-', lw=1.2, label='published bed')
    row = iy_all == IY
    ax.plot(ix_all[row] * PX_M / 1000, vals_all[row], 'k.', ms=6, label='radar (gridded, aligned)')
    ax.legend(loc='upper right', fontsize=8); ax.set_ylabel('bed z (m)')
axs[-1].set_xlabel('along-flow x (km)'); axs[0].set_title(f'flowline cross-section, row iy={IY}')
fig.savefig(FIGS / 'mgpl_xsection.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# --- one-figure summary of the generators -------------------------------------
fig, axs = plt.subplots(1, 4, figsize=(19, 4.6), constrained_layout=True)
panels = [(trend, 'published bed'), (sgs_beds[0], 'MGSIM package — global variogram (nb 03)'),
          (mgpl_beds[0], 'MGSIM package — per-level variogram (nb 05)'), (grf_beds[0], 'conditional GRF (nb 04)')]
for ax, (b, t) in zip(axs, panels):
    im = ax.imshow(b, origin='lower', cmap='terrain', vmin=vmin, vmax=vmax)
    ax.set_title(t, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axs, shrink=0.8, label='bed z (m)')
fig.savefig(FIGS / 'methods_compare_all.png', dpi=110, bbox_inches='tight')
plt.show()
print('(the kriging-demo port is in notebook 04; it is not an Elmer candidate)')

## Reading the result

The schedule transferred faithfully — the fitted range steps down 2.04 → 1.27 →
0.57 → 0.27 km and the residual at the data shrinks 57 → 52 → 40 → 25 m, nearly
identically across all ten members (range std ≤ 0.1 km). And it makes the bed
**worse on every metric but one**: roughness 62 m (global variogram 30 m,
conditional GRF 5 m, published bed 3.7 m), departure-from-published std 80 m
(data: 57 m), ensemble spread above the global-variogram run at every distance
(28 m floor on the lines, 82 m plateau). Only the conditioning median improves
(5.0 m vs 9.5 m), because the finest level reaches the data with a near-node
range.

Why, in one sentence: each level's NST and variogram are defined by the residual
*at the data*, but the SGS then paints that residual's full variance over the
**whole grid** at a range shorter than our gaps — and with 80% of nodes more than
a range from any line, the fine passes are effectively unconditional short-range
noise stacked on top of the coarse pass. The multipass variance inflation of
notebook 03 (a sum of level sills) is untouched; the refit only shortens the
wavelength of what gets added.

WestPilbara never sees this because its lines are four cells apart: every node
is within range of a datum at every level, so the fine-level SGS variance is
tiny and each pass behaves like interpolation. The smoothness of those maps is a
property of the survey geometry, not of the per-level schedule.

Consequence for the Elmer step: notebook 03 (package, global variogram) stays
the MGSIM default; the per-level set is saved (`*_mgpl??.dat`) for completeness
but not recommended; the conditional-GRF hybrid remains the smooth-and-honest
alternative.

In [ ]:
# --- save: npz + sif-format DEMs (gitignored) ---------------------------------
meta = dict(method='multigrid_sgsim package, one pass per level with NST+variogram refit per level (WestPilbara schedule)',
            trend='published bed DEM (source_copies)', datum_shift_m=datum_shift, trim_pct=[0.25, 99.75],
            conditioning='checkerboard-thinned obs, as nb 03', mg_levels_px=MG_LEVELS, num_points=K_NEIGHBORS,
            radius_px=RADIUS, clip_percentile=CLIP_PCT, nugget_rel=0.01, seeds=list(range(N_REAL)),
            per_level_info=infos, taper='nb-03 cosine 2 km to published bed', min_ice_m=MIN_ICE)
np.savez_compressed(DERIVED / 'bed_realizations_cw1951_mgpl.npz', beds=mgpl_beds, trend=trend, surface=surface,
                    taper=taper_w, obs_mask=obs_mask, meta=json.dumps(meta))
xy = np.loadtxt(SRC / '2019cw428_bed_d25km_h1004_a0.792.dat')[:, :2]
dem_dir = DERIVED / 'dems'; dem_dir.mkdir(exist_ok=True)
for i, bed in enumerate(mgpl_beds):
    np.savetxt(dem_dir / f'2019cw428_bed_d25km_h1004_a0.792_mgpl{i:02d}.dat',
               np.column_stack([xy, bed.ravel()]), fmt='%f %f %f')
print(f'saved bed_realizations_cw1951_mgpl.npz + {N_REAL} DEMs (*_mgpl??.dat) to {dem_dir}/')